# Multiple-comparison control across all hypothesis tests

The paper reports 42 hypothesis tests across the demographics and RQ1–RQ3. This notebook recomputes each of them exactly as the RQ notebooks do and controls the **false discovery rate over the whole family** with the Benjamini–Hochberg (BH) procedure.

A result is treated as statistically significant only if $q < 0.05$, and as exploratory if $p < 0.05$ but $q \ge 0.05$. Dunn's post-hoc pairs (Bonferroni-corrected within their omnibus test) and the descriptive phase–phase correlations are not part of the family.

Output: `multiple_comparisons.csv`.

In [1]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from analysis_utils import load_data, DS_CHALLENGE_COLS, DS_CHALLENGE_LABELS

warnings.filterwarnings('ignore')
df = load_data()
print(f"Dataset loaded: {len(df)} respondents")

PHASES = {'phase_frequency_header_freq_business_understanding': 'Business Understanding',
          'freq_data_understanding': 'Data Understanding', 'freq_data_preparation': 'Data Preparation',
          'freq_modeling': 'Modeling', 'freq_evaluation': 'Evaluation', 'freq_deployment': 'Deployment'}
rows = []

def kw_by(frame, col, group='role', min_n=3):
    sub = frame[[col, group]].dropna()
    groups = [g[col].values for _, g in sub.groupby(group) if len(g) >= min_n]
    h, p = stats.kruskal(*groups)
    return h, p, len(sub)

def add(section, test, variables, n, stat, p):
    rows.append({'section': section, 'test': test, 'variables': variables,
                 'n': n, 'statistic': round(stat, 3), 'p': p})

Dataset loaded: 112 respondents


## The 42 tests

In [2]:
# Demographics: Kendall's tau-b between DS and SE experience ('no SE experience' merged into '<1 year')
tau, p = stats.kendalltau(df['ds_exp'], df['se_exp'].replace({0: 1}))
add('Demographics', "Kendall tau_b", 'DS experience x SE experience', len(df), tau, p)

# RQ1: phase frequency by role
for col, lab in PHASES.items():
    h, p, n = kw_by(df, col)
    add('RQ1', 'Kruskal-Wallis H', f'{lab} frequency by role', n, h, p)

# RQ2: challenge frequency by role, and experience x challenge frequency
for c in DS_CHALLENGE_COLS:
    h, p, n = kw_by(df, c)
    add('RQ2', 'Kruskal-Wallis H', f'{DS_CHALLENGE_LABELS[c]} by role', n, h, p)
for exp, lab in [('ds_exp', 'DS experience'), ('se_exp', 'SE experience')]:
    for c in DS_CHALLENGE_COLS:
        t = df[[exp, c]].dropna()
        r, p = stats.spearmanr(t[exp], t[c])
        add('RQ2', 'Spearman rho', f'{lab} x {DS_CHALLENGE_LABELS[c]}', len(t), r, p)

# RQ3: SE adoption score (all respondents)
for exp, lab in [('se_exp', 'SE experience'), ('ds_exp', 'DS experience')]:
    t = df[[exp, 'adoption_score']].dropna()
    r, p = stats.spearmanr(t[exp], t['adoption_score'])
    add('RQ3', 'Spearman rho', f'{lab} x adoption score', len(t), r, p)
for col, lab in [('role', 'role'), ('organization_type_Response', 'organization type'),
                 ('knows_ds_framework_Response', 'DS framework knowledge')]:
    h, p, n = kw_by(df, 'adoption_score', group=col)
    add('RQ3', 'Kruskal-Wallis H', f'adoption score by {lab}', n, h, p)

tests = pd.DataFrame(rows)
print(f"{len(tests)} tests")

42 tests


## Benjamini–Hochberg FDR correction over the whole family

In [3]:
def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order] * m / np.arange(1, m + 1)
    q = np.empty(m)
    q[order] = np.minimum.accumulate(ranked[::-1])[::-1]
    return np.minimum(q, 1.0)

tests['q'] = benjamini_hochberg(tests['p'])
tests['verdict'] = np.where(tests['q'] < 0.05, 'significant',
                   np.where(tests['p'] < 0.05, 'exploratory', 'not significant'))
tests.to_csv('multiple_comparisons.csv', index=False)

pd.set_option('display.width', 200)
display(tests.assign(p=tests['p'].map('{:.4g}'.format), q=tests['q'].map('{:.4g}'.format)))

,section,test,variables,n,statistic,p,q,verdict
0,Demographics,Kendall tau_b,DS experience x SE experience,112,0.510,6.858e-10,2.88e-08,significant
1,RQ1,Kruskal-Wallis H,Business Understanding frequency by role,87,6.671,0.3524,0.7736,not significant
2,RQ1,Kruskal-Wallis H,Data Understanding frequency by role,86,14.184,0.02765,0.3871,exploratory
3,RQ1,Kruskal-Wallis H,Data Preparation frequency by role,87,9.933,0.1275,0.595,not significant
4,RQ1,Kruskal-Wallis H,Modeling frequency by role,87,3.911,0.6887,0.8765,not significant
5,RQ1,Kruskal-Wallis H,Evaluation frequency by role,86,1.415,0.965,0.9885,not significant
6,RQ1,Kruskal-Wallis H,Deployment frequency by role,87,4.970,0.5477,0.8684,not significant
7,RQ2,Kruskal-Wallis H,High learning curve of business [BU] by role,87,5.206,0.5177,0.8684,not significant
8,RQ2,Kruskal-Wallis H,Misunderstanding DS/users [BU] by role,87,3.185,0.7853,0.9423,not significant
9,RQ2,Kruskal-Wallis H,New/unplanned feature requests [BU] by role,87,4.111,0.6616,0.8684,not significant


In [4]:
display(tests[tests['p'] < 0.05].sort_values('p')[['section', 'variables', 'statistic', 'p', 'q', 'verdict']])

,section,variables,statistic,p,q,verdict
0,Demographics,DS experience x SE experience,0.510,6.857622e-10,2.880201e-08,significant
37,RQ3,SE experience x adoption score,0.247,8.656347e-03,1.817833e-01,exploratory
2,RQ1,Data Understanding frequency by role,14.184,2.765150e-02,3.871211e-01,exploratory
13,RQ2,Learning curve in data exploration [DU] by role,12.888,4.485704e-02,4.709990e-01,exploratory
